In [2]:
import pandas as pd

In [9]:
def count_words(stroke: str): #функция подсчета слов без повторов
  count = len(set(map(lambda x: x.strip(), stroke.split(','))))
  return count

In [ ]:
def count(stroke: str): #функция подсчета слов с повторами
  count = len(list(map(lambda x: x.strip(), stroke.split(','))))
  return count

In [ ]:
def avge(s_n, s_r): #функция подсчета процента повторов
  rep = s_r - s_n
  return rep / s_r

####Традиционные параметры

подсчет характеристик на датасете words проводился следующим образом для:


*  средней длины ответа в словах
*  средней длины ответа в словах с учетом повторений
*  процент повторений
*  процент



In [4]:
words = pd.read_csv("####", encoding='cp1251', sep=";") #считывается датафрейм с данными пациентов

Подсчет средней длины ответа в словах без повторений

In [ ]:
words['S1_count'] = words['VFS1'].apply(count_words)
words['S2_count'] = words['VFS2'].apply(count_words)
words['S3_count'] = words['VFS3'].apply(count_words)

In [ ]:
words['avg_wc'] = (words['S1_count'] + words['S2_count'] + words['S3_count']) / 3

Подсчет средней длины ответа пациента в словах с повторениями

In [ ]:
words['S1_count_rep'] = words['VFS1'].apply(count)
words['S2_count_rep'] = words['VFS2'].apply(count)
words['S3_count_rep'] = words['VFS3'].apply(count)

Подсчет процента повторов

In [ ]:
words['proc_1'] = words.apply(lambda row: avge(row['S1_count'], row['S1_count_rep']), axis=1)
words['proc_2'] = words.apply(lambda row: avge(row['S2_count'], row['S2_count_rep']), axis=1)
words['proc_3'] = words.apply(lambda row: avge(row['S3_count'], row['S3_count_rep']), axis=1)


In [ ]:
words['avg_rep'] = ((words['S1_count_rep']) + (words['S2_count_rep']) + (words['S3_count_rep'])) / 3 #средняя длина ответа с повторениями
words['percentage'] = (words['proc_1'] + words['proc_2'] + words['proc_3']) / 3 #средний процент
words['irr'] = words['S1_Irr_words'] + words['S2_Irr_words'] #количество нерелевантных слов


####Алгоритм лемматизации

В основе лемматизатор PyMorphy3, который возращает как начальную форму существительного, так и форму виду "начальнаяформа_NOUN", необходимую для использования Word2Vec.

В случае, когда названный объект состоял из двух слов, лемматизировалось последнее, так как во всех случаях это названия вида "прил + сущ." (прим. красная смородина, цветная капуста), где основную смысловую нагрузку несет именно вершина-существительное

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import gensim
import gensim.downloader
import pymorphy3

print("Загрузка модели w2v...") #тяжелые модельки...
morph = pymorphy3.MorphAnalyzer()
w2v = gensim.downloader.load('word2vec-ruscorpora-300')
print("Модель загружена.")

In [ ]:
def lemmatize(stroke: str):
  if pd.isna(stroke) or stroke == ' ':
      return [[], []]
  else:

    listik = list(map(lambda x: x.strip(), stroke.split(',')))
    lemmed = []
    posed = []
    for word in listik:
        if " " in word:
          word = word.split(' ')[-1]
        #исправление ошибок разметки
        #генерализация лексем для избегания нулевых значений при векторизации
        if word == 'личи':
          word_p = 'слива_NOUN'

        elif word == 'цукини':
          word_p = 'кабачок_NOUN'
        elif word == 'дайкон':
          word_p = 'редька_NOUN'
        elif word == 'барабарис':
          word_p = 'шиповник_NOUN'
        elif word == 'кальраби':
          word_p = 'капуста_NOUN'
        elif word == 'нектарин':
          word_p = 'персик_NOUN'
        elif word == 'маракуйя':
          word_p = 'персик_NOUN'

        elif word == 'альпака':
          word_p = 'лама_NOUN'
        elif word == 'аноилопа':
          word_p = 'антилопа_NOUN'
        elif word == 'бегмот':
          word_p = 'бегемот_NOUN'
        elif word == 'спасан':
          word_p = 'сапсан_NOUN'


        else:
          p = morph.parse(word)[0]
          word = p.normal_form
          word_p = word + "_" + p.tag.POS

        lemmed.append(word.replace('ё', 'е'))
        posed.append(word_p.replace('ё', 'е'))
    return [lemmed, posed]

In [ ]:
words['S1_lem'] = words['VFS1'].apply(lemmatize)
words['S2_lem'] = words['VFS2'].apply(lemmatize)
words['S3_lem'] = words['VFS3'].apply(lemmatize)

####Алгоритм векторизации

В предобученный на корпусе word2vec-ruscorpora-300 векторизатор Word2Vec на вход подается список слов вида"начальнаяформа_NOUN", на выходе список векторных представленний этих слов длины (300,) каждый

In [ ]:
def vectorize_text(text: list): #только для русского языка
    res = []
    for word in text:
        if word in w2v:
            res.append(w2v[word])
        else:
            print(f"Слово не найдено в w2v: '{word}'")
            res.append(np.zeros(300))
    if len(res) == 0:
        return np.zeros(300)
    else:
        return np.vstack(res)

In [ ]:
words['S1_vec'] = words['S1_lem'].apply(lambda x: vectorize_text(x[1]))
words['S2_vec'] = words['S2_lem'].apply(lambda x: vectorize_text(x[1]))
words['S3_vec'] = words['S3_lem'].apply(lambda x: vectorize_text(x[1]))

####Алгоритм кластеризации

Алгоритм кластеризации подробно описан в тексте курсовой работы и опирается на расчет расстояния до центроида (Toro-Hernández и др. 2026).

In [ ]:
def count_switches_lin(X, listik): #по методу центроида
    threshold = 0.565
    centroid = X[0]
    cluster_ids = [0]
    current_cluster = 0
    switch_words = []
    clusters = [[listik[0]]]
    for i in range(1, X.shape[0]):
        sim = np.dot(X[i], centroid)
        if sim < threshold:
            current_cluster += 1
            switch_words.append(listik[i])
            clusters.append([listik[i]])
            centroid = X[i]
        else:
            clusters[current_cluster].append(listik[i])
            centroid = (centroid + X[i]) / 2
        cluster_ids.append(current_cluster)
    return clusters

In [ ]:
words['S1_clust_centr'] = words.apply(
    lambda row: count_switches_lin(row['S1_vec'], row['S1_lem'][0]),
    axis=1
)

In [ ]:
words['S2_clust_centr'] = words.apply(
    lambda row: count_switches_lin(row['S2_vec'], row['S2_lem'][0]),
    axis=1
)

In [ ]:
words['S3_clust_centr'] = words.apply(
    lambda row: count_switches_lin_an(row['S3_vec'], row['S3_lem'][0]),
    axis=1
)

####Расчет кластерных характеристик ответов

Предварительно рассчитывается значение длины в кластерах и количество переключений для каждого ответа

In [ ]:
words['s1_cl_c_len'] = words.apply(lambda x: len(x['S1_clust_centr']), axis=1)
words['s1_cl_c_len_sw'] = words.apply(lambda x: len(x['S1_clust_centr']) - 1, axis=1)

In [ ]:
words['s2_cl_c_len'] = words.apply(lambda x: len(x['S2_clust_centr']), axis=1)
words['s2_cl_c_len_sw'] = words.apply(lambda x: len(x['S2_clust_centr']) - 1, axis=1)

In [ ]:
words['s3_cl_c_len'] = words.apply(lambda x: len(x['S3_clust_centr']), axis=1)
words['s3_cl_c_len_sw'] = words.apply(lambda x: len(x['S3_clust_centr']) - 1, axis=1)

Расчет **среднего размера кластера** по трем категориям: сумма длин всех кластеров резделить на их общее количество

In [ ]:
def avg_claster_length(row):
    total_sum = (
        sum(len(i) for i in row['S1_clust_centr']) +
        sum(len(i) for i in row['S2_clust_centr']) +
        sum(len(i) for i in row['S3_clust_centr'])
    )

    total_count = (
        row['s1_cl_c_len'] +
        row['s2_cl_c_len'] +
        row['s3_cl_c_len']
    )

    return total_sum / total_count if total_count != 0 else 0

In [ ]:
words['avg_each_cluster_len'] = words.apply(avg_claster_length, axis=1)

Расчет **количества кластеров и количества переключений на 100 слов** произведен как сумма значений по трем категориям, разделенная на общее количество слов и умноженная на 100

In [ ]:
words['avg_cl_len_norm'] = ( #количество кластеров на 100 слов
    words['s1_cl_c_len'] +
    words['s2_cl_c_len'] +
    words['s3_cl_c_len']
) / (
    words['S1_lem'][0].apply(len) +
    words['S2_lem'][0].apply(len) +
    words['S3_lem'][0].apply(len)
) * 100

In [ ]:
words['avg_cl_sw_norm'] = (
    words['s1_cl_c_len_sw'] + #количество переключений на 100 слов
    words['s2_cl_c_len_sw'] +
    words['s3_cl_c_len_sw']
) / (
    words['S1_lem'].apply(len) +
    words['S2_lem'].apply(len) +
    words['S3_lem'].apply(len)
) * 100

Расчет **средней длины высказывания в кластерах** для 2 категорий ("овощи"/"фрукты") и 3 категорий

In [ ]:
words['cl1'] = (words['s1_cl_c_len'] + words['s2_cl_c_len']) / 2 #среднее длина без "животные"
words['avg_cl_len'] = (words['s1_cl_c_len'] + words['s2_cl_c_len'] + words['s3_cl_c_len']) / 3 #одобрено